# Beta Scan VAE Training - Using Existing Training Infrastructure
Systematic beta scanning with the existing training function:
- Start with beta=1e-5
- Multiply by 3 each iteration
- Load weights from previous beta run
- Continue until KL < 10 at early stopping

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import time
import nibabel as nib
import json

# Import existing training infrastructure
from data.data_ingestion import collect_files, generate_dataframe
from data.dataloader import create_dataloaders
from models.vae.vae_bottleneck_models import (
    BottleneckEncoder,
    ComplexBottleneckDeconvDecoder,
    BottleneckVAE,
)
from models.vae.training import train_vae
from models.vae.config import VAEConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Define Weighted VAE Loss Function

In [ ]:
class WeightedVAELoss(nn.Module):
    """
    Weighted VAE Loss with Luca's improvements:
    - Fixes 3D Axis Mismatch using permute(2, 1, 0)
    - Removes weight normalization
    - Uses 0.1 padding for background
    - Implements Beta Warmup from 0 to beta_final
    """
    def __init__(self, weight_mask_path, beta=0.0005, beta_warmup_steps=5000):
        super().__init__()
        
        # Load the Mask
        weight_nii = nib.load(weight_mask_path)
        weight_data = torch.from_numpy(weight_nii.get_fdata()).float()
        
        # Fix 3D axis swap
        weight_data = weight_data.permute(2, 1, 0)
        
        # Add batch/channel dims: [1, 1, D, H, W]
        self.full_mask = weight_data.unsqueeze(0).unsqueeze(0)
        
        # No normalization (per Luca)
        
        # Register as buffer
        self.register_buffer('weight_mask', self.full_mask)
        
        self.beta = beta
        self.beta_warmup_steps = beta_warmup_steps
        self.current_step = 0
        
        print(f"✅ Loss Initialized. Mask Permuted (2,1,0). Shape: {self.full_mask.shape}")
        print(f"   Beta: 0 → {beta} over {beta_warmup_steps} steps")
        
    def smart_adjust_mask(self, mask, target_shape):
        """
        Adjusts mask to match the input batch size (D, H, W).
        Uses padding value 0.1 for background.
        """
        _, _, D_cur, H_cur, W_cur = mask.shape
        D_tgt, H_tgt, W_tgt = target_shape
        
        # PAD if smaller
        pad_d = max(0, D_tgt - D_cur)
        pad_h = max(0, H_tgt - H_cur)
        pad_w = max(0, W_tgt - W_cur)
        
        if pad_d > 0 or pad_h > 0 or pad_w > 0:
            mask = F.pad(mask, (pad_w//2, pad_w-pad_w//2, 
                                pad_h//2, pad_h-pad_h//2, 
                                pad_d//2, pad_d-pad_d//2), 
                         mode='constant', value=0.1)

        # CROP if larger
        _, _, D_new, H_new, W_new = mask.shape
        d_start = (D_new - D_tgt) // 2
        h_start = (H_new - H_tgt) // 2
        w_start = (W_new - W_tgt) // 2
        
        return mask[:, :, d_start:d_start+D_tgt, h_start:h_start+H_tgt, w_start:w_start+W_tgt]
    
    def __call__(self, recon_x, x, mu, log_var):
        # Ensure mask is on correct device
        if self.weight_mask.device != recon_x.device:
            self.weight_mask = self.weight_mask.to(recon_x.device)
            
        current_target_shape = recon_x.shape[2:] 
        
        # Adjust mask to current batch dimensions if needed
        if self.weight_mask.shape[2:] != current_target_shape:
            adjusted_mask = self.smart_adjust_mask(self.weight_mask, current_target_shape)
        else:
            adjusted_mask = self.weight_mask
            
        # Weighted Reconstruction Loss
        squared_error = (recon_x - x) ** 2
        weighted_error = squared_error * adjusted_mask
        recon_loss = weighted_error.mean()
        
        # KL Divergence
        kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1)
        kl_loss = torch.mean(kl_loss)
        
        # Beta Warmup (cycling benefit with initial_beta=0)
        if self.current_step < self.beta_warmup_steps:
            beta = self.beta * (self.current_step / self.beta_warmup_steps)
        else:
            beta = self.beta
            
        self.current_step += 1
        
        # Total Loss
        total_loss = recon_loss + (beta * kl_loss)
        
        return total_loss, recon_loss, kl_loss, beta

## Data Setup

In [ ]:
# Configuration
data_dir = "data/Images"
mask_path = "data/masks/rmask_ICV.nii"
batch_size = 4
output_dir = "output/Experiments/BetaScanVAE"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Prepare data
print("Collecting files...")
included_files, excluded_files = collect_files(data_dir)
print(f"Found {len(included_files)} valid files, excluded {len(excluded_files)} files")

# Generate dataframe
print("Generating dataframe...")
df = generate_dataframe(included_files)
display(df.head())

# Create dataloaders - FIX: Set num_workers=0 for Jupyter notebooks
print("Creating dataloaders...")
train_loader, val_loader = create_dataloaders(
    df, batch_size=batch_size, train_split=0.8, 
    on_demand=True, mask_path=mask_path,
    num_workers=0  # Important: 0 for Jupyter to avoid multiprocessing issues
)

print(f"Data preparation complete. Train: {len(train_loader.dataset)}, Val: {len(val_loader.dataset)}")

## Beta Scan Configuration

In [ ]:
# Beta scan parameters
beta_values = [1e-5, 3e-5, 9e-5, 2.7e-4, 8.1e-4, 2.43e-3, 7.29e-3]
current_beta_index = 0  # Change this to continue from a specific beta
current_beta = beta_values[current_beta_index]

# Directory to store beta scan results
beta_scan_dir = os.path.join(output_dir, "beta_scan_results")
os.makedirs(beta_scan_dir, exist_ok=True)

# Metrics storage for all beta runs
beta_scan_results = {
    'beta': [],
    'final_train_recon': [],
    'final_train_kl': [],
    'final_val_recon': [],
    'final_val_kl': [],
    'final_beta_kl': [],
    'epochs_trained': []
}

# Try to load existing results
summary_path = os.path.join(beta_scan_dir, "beta_scan_summary.json")
if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        beta_scan_results = json.load(f)
    print(f"✓ Loaded existing beta scan results: {len(beta_scan_results['beta'])} runs completed")

print(f"\nBeta scan configured: {beta_values}")
print(f"Starting with beta = {current_beta:.2e}")
print(f"Results directory: {beta_scan_dir}")

## Model Setup

In [ ]:
target_shape = (64, 128, 128)
latent_dim = 256

def build_vae_model(latent_dim=256):
    """Build BottleneckVAE model"""
    model = BottleneckVAE(
        BottleneckEncoder(initial_filters=4, latent_dim=latent_dim, bottleneck_shape=(1,1,1)),
        ComplexBottleneckDeconvDecoder(latent_dim=latent_dim, target_shape=target_shape),
        latent_dim=latent_dim
    )
    return model.to(device)

model = build_vae_model(latent_dim=latent_dim)

# Load weights from previous beta run if available
if current_beta_index > 0:
    prev_beta = beta_values[current_beta_index - 1]
    prev_checkpoint = os.path.join(beta_scan_dir, f"beta_{prev_beta:.2e}_best.pth")
    if os.path.exists(prev_checkpoint):
        model.load_state_dict(torch.load(prev_checkpoint, map_location=device))
        print(f"✓ Loaded weights from previous beta run: {prev_checkpoint}")
    else:
        print(f"⚠ Previous checkpoint not found: {prev_checkpoint}")
        print("Training from scratch...")
else:
    print("Starting first beta run from scratch")

num_params = count_trainable(model)
print(f"Model Parameters: {num_params / 1e6:.2f}M")

## Replace VAELoss in training.py with WeightedVAELoss

In [ ]:
# Monkey patch the VAELoss import in the training module
import models.vae.training as training_module
training_module.VAELoss = WeightedVAELoss

print("✓ Replaced VAELoss with WeightedVAELoss in training module")

## Training Configuration

In [ ]:
# Create VAE config for current beta
config = VAEConfig(
    latent_dim=latent_dim,
    learning_rate=1e-4,
    batch_size=batch_size,
    accumulation_steps=1,  # No accumulation since we're using batch_size=4
    epochs=150,
    early_stopping_patience=20,
    beta=current_beta,
    beta_warmup_steps=5000,
    use_mixed_precision=True,
    gradient_clip=1.0,
    num_workers=0,  # Important for Jupyter
    checkpoint_dir=os.path.join(beta_scan_dir, f"beta_{current_beta:.2e}"),
    model_name=f"beta_{current_beta:.2e}",
    save_interval=10
)

# Pass weight_mask_path to the loss function
# This will be used when VAELoss is instantiated in train_vae
config.weight_mask_path = "data/masks/weightMatrix.nii"

## Custom Training Function with WeightedVAELoss

In [ ]:
# We need to modify the train_vae function to use WeightedVAELoss
# Let's create a wrapper that initializes the criterion correctly

from torch.cuda import amp
from models.vae.optimizer import create_vae_optimizer
from models.vae.scheduler import create_vae_scheduler
from models.vae.config import VAECheckpointHandler, VAEEarlyStopping
from tqdm import tqdm
import gc

def train_vae_beta_scan(model, train_loader, val_loader, config):
    """Modified training function using WeightedVAELoss"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Initialize WeightedVAELoss instead of VAELoss
    criterion = WeightedVAELoss(
        weight_mask_path=config.weight_mask_path,
        beta=config.beta,
        beta_warmup_steps=config.beta_warmup_steps
    )
    
    optimizer = create_vae_optimizer(model, config)
    scheduler = create_vae_scheduler(optimizer, config)
    early_stopping = VAEEarlyStopping(patience=config.early_stopping_patience)
    checkpoint_handler = VAECheckpointHandler(config.checkpoint_dir, config.model_name)

    # Mixed precision setup
    scaler = amp.GradScaler(enabled=config.use_mixed_precision)

    # Training tracking variables
    train_losses = []
    val_losses = []
    train_recon_losses = []
    train_kl_losses = []
    val_recon_losses = []
    val_kl_losses = []
    best_val_loss = float('inf')
    start_time = time.time()

    # Training loop
    print(f"\n{'='*60}")
    print(f"Starting training for beta = {config.beta:.2e}")
    print(f"{'='*60}\n")
    
    for epoch in range(config.epochs):
        # Training phase
        model.train()
        epoch_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0
        current_beta = 0

        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.epochs} [Train]', leave=True)

        for batch_idx, batch in enumerate(train_pbar):
            volumes = batch['volume'].to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            # Forward pass
            with amp.autocast(enabled=config.use_mixed_precision):
                reconstructed, mu, log_var = model(volumes)
                loss, recon_loss, kl_loss, beta = criterion(reconstructed, volumes, mu, log_var)
                current_beta = beta

            # Backward pass
            scaler.scale(loss).backward()
            
            if config.gradient_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip)
            
            scaler.step(optimizer)
            scaler.update()

            # Track loss
            epoch_loss += loss.item()
            epoch_recon_loss += recon_loss.item()
            epoch_kl_loss += kl_loss.item()
            
            train_pbar.set_postfix({
                'loss': f"{loss.item():.6f}",
                'recon': f"{recon_loss.item():.6f}",
                'kl': f"{kl_loss.item():.2f}",
                'beta': f"{beta:.6f}"
            })

            del volumes, reconstructed, mu, log_var, loss, recon_loss, kl_loss

        # Calculate average training losses
        avg_train_loss = epoch_loss / len(train_loader)
        avg_train_recon_loss = epoch_recon_loss / len(train_loader)
        avg_train_kl_loss = epoch_kl_loss / len(train_loader)
        
        train_losses.append(avg_train_loss)
        train_recon_losses.append(avg_train_recon_loss)
        train_kl_losses.append(avg_train_kl_loss)

        # Validation phase
        model.eval()
        val_loss = 0
        val_recon_loss_sum = 0
        val_kl_loss_sum = 0
        
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{config.epochs} [Val]', leave=True)

        with torch.no_grad():
            for batch in val_pbar:
                volumes = batch['volume'].to(device)
                reconstructed, mu, log_var = model(volumes)
                loss, recon_loss, kl_loss, _ = criterion(reconstructed, volumes, mu, log_var)
                
                val_loss += loss.item()
                val_recon_loss_sum += recon_loss.item()
                val_kl_loss_sum += kl_loss.item()
                
                val_pbar.set_postfix({
                    'loss': f"{loss.item():.6f}",
                    'recon': f"{recon_loss.item():.6f}",
                    'kl': f"{kl_loss.item():.2f}"
                })

                del volumes, reconstructed, mu, log_var, loss, recon_loss, kl_loss

        # Calculate average validation losses
        avg_val_loss = val_loss / len(val_loader)
        avg_val_recon_loss = val_recon_loss_sum / len(val_loader)
        avg_val_kl_loss = val_kl_loss_sum / len(val_loader)
        
        val_losses.append(avg_val_loss)
        val_recon_losses.append(avg_val_recon_loss)
        val_kl_losses.append(avg_val_kl_loss)

        # Update learning rate
        scheduler.step(avg_val_loss)

        # Check if this is the best model
        is_best = avg_val_loss < best_val_loss
        if is_best:
            best_val_loss = avg_val_loss
            # Save best model to beta_scan_dir root
            best_model_path = os.path.join(beta_scan_dir, f"beta_{config.beta:.2e}_best.pth")
            torch.save(model.state_dict(), best_model_path)

        # Save checkpoint
        if (epoch + 1) % config.save_interval == 0 or is_best:
            checkpoint_handler.save(
                model, optimizer, scheduler,
                epoch, train_losses, val_losses, 
                train_recon_losses, train_kl_losses,
                val_recon_losses, val_kl_losses,
                is_best=is_best
            )

        # Print epoch summary
        print(f"\nEpoch {epoch+1}/{config.epochs}:")
        print(f"Train Loss: {avg_train_loss:.6f} (Recon: {avg_train_recon_loss:.6f}, KL: {avg_train_kl_loss:.2f})")
        print(f"Val Loss: {avg_val_loss:.6f} (Recon: {avg_val_recon_loss:.6f}, KL: {avg_val_kl_loss:.2f})")
        print(f"Beta*KL: {config.beta * avg_val_kl_loss:.6f}")
        print(f"LR: {optimizer.param_groups[0]['lr']:.8f}\n")
        
        # Early stopping check
        early_stopping(avg_val_loss, epoch)
        if early_stopping.early_stop:
            print("\nEarly stopping triggered!")
            break

    total_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"✓ Training completed in {total_time:.1f}s ({total_time/60:.1f} min)")
    print(f"Best validation loss: {best_val_loss:.6f}")
    print(f"Final validation KL: {val_kl_losses[-1]:.2f}")
    print(f"{'='*60}\n")
    
    return {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_recon_losses': train_recon_losses,
        'train_kl_losses': train_kl_losses,
        'val_recon_losses': val_recon_losses,
        'val_kl_losses': val_kl_losses,
        'best_val_loss': best_val_loss,
        'model': model
    }

## Run Training

In [ ]:
# Train the model
results = train_vae_beta_scan(model, train_loader, val_loader, config)

# Store results for this beta
beta_scan_results['beta'].append(current_beta)
beta_scan_results['final_train_recon'].append(results['train_recon_losses'][-1])
beta_scan_results['final_train_kl'].append(results['train_kl_losses'][-1])
beta_scan_results['final_val_recon'].append(results['val_recon_losses'][-1])
beta_scan_results['final_val_kl'].append(results['val_kl_losses'][-1])
beta_scan_results['final_beta_kl'].append(current_beta * results['val_kl_losses'][-1])
beta_scan_results['epochs_trained'].append(len(results['train_losses']))

# Save beta scan summary
with open(summary_path, 'w') as f:
    json.dump(beta_scan_results, f, indent=2)
print(f"✓ Beta scan summary saved: {summary_path}")

# Decision guidance
final_kl = results['val_kl_losses'][-1]
print(f"\n{'='*60}")
print("NEXT STEPS:")
if final_kl > 10:
    print(f"✓ KL={final_kl:.2f} > 10: Continue to next beta")
    print(f"  Set current_beta_index = {current_beta_index + 1}")
    if current_beta_index + 1 < len(beta_values):
        print(f"  Next beta: {beta_values[current_beta_index + 1]:.2e}")
else:
    print(f"✓ KL={final_kl:.2f} < 10: Beta scan complete!")
    print("  Proceed to analysis and visualization")
print(f"{'='*60}\n")

## Beta Scan Summary and Analysis

In [ ]:
# Load beta scan summary
if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        beta_scan_results = json.load(f)
    
    print("Beta Scan Results Summary:")
    print("="*80)
    print(f"{'Beta':<12} {'Val Recon':<12} {'Val KL':<10} {'Beta*KL':<12} {'Epochs':<8}")
    print("="*80)
    for i, beta in enumerate(beta_scan_results['beta']):
        print(f"{beta:<12.2e} {beta_scan_results['final_val_recon'][i]:<12.6f} "
              f"{beta_scan_results['final_val_kl'][i]:<10.2f} "
              f"{beta_scan_results['final_beta_kl'][i]:<12.6f} "
              f"{beta_scan_results['epochs_trained'][i]:<8}")
    print("="*80)
else:
    print("No beta scan summary found. Run training first.")

## Beta Scan Visualization

In [ ]:
# Plot beta scan results across all beta values
if len(beta_scan_results['beta']) > 1:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    betas = beta_scan_results['beta']
    
    # Plot 1: Reconstruction Error vs Beta
    axes[0, 0].plot(betas, beta_scan_results['final_train_recon'], 'o-', label='Train Recon', linewidth=2, markersize=8)
    axes[0, 0].plot(betas, beta_scan_results['final_val_recon'], 's-', label='Val Recon', linewidth=2, markersize=8)
    axes[0, 0].set_xlabel('Beta', fontsize=12)
    axes[0, 0].set_ylabel('Reconstruction Error', fontsize=12)
    axes[0, 0].set_xscale('log')
    axes[0, 0].set_title('Reconstruction Error vs Beta', fontsize=14, fontweight='bold')
    axes[0, 0].legend(fontsize=10)
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: KL Divergence vs Beta
    axes[0, 1].plot(betas, beta_scan_results['final_train_kl'], 'o-', label='Train KL', linewidth=2, markersize=8)
    axes[0, 1].plot(betas, beta_scan_results['final_val_kl'], 's-', label='Val KL', linewidth=2, markersize=8)
    axes[0, 1].axhline(y=10, color='r', linestyle='--', linewidth=2, label='KL=10 threshold')
    axes[0, 1].set_xlabel('Beta', fontsize=12)
    axes[0, 1].set_ylabel('KL Divergence', fontsize=12)
    axes[0, 1].set_xscale('log')
    axes[0, 1].set_title('KL Divergence vs Beta', fontsize=14, fontweight='bold')
    axes[0, 1].legend(fontsize=10)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Beta * KL vs Beta
    axes[1, 0].plot(betas, beta_scan_results['final_beta_kl'], 'o-', color='purple', linewidth=2, markersize=8)
    axes[1, 0].set_xlabel('Beta', fontsize=12)
    axes[1, 0].set_ylabel('Beta * KL', fontsize=12)
    axes[1, 0].set_xscale('log')
    axes[1, 0].set_title('Beta * KL vs Beta', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Epochs Trained vs Beta
    axes[1, 1].bar(range(len(betas)), beta_scan_results['epochs_trained'], color='teal', alpha=0.7)
    axes[1, 1].set_xlabel('Beta', fontsize=12)
    axes[1, 1].set_ylabel('Epochs Trained', fontsize=12)
    axes[1, 1].set_title('Training Duration per Beta', fontsize=14, fontweight='bold')
    axes[1, 1].set_xticks(range(len(betas)))
    axes[1, 1].set_xticklabels([f"{b:.2e}" for b in betas], rotation=45)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plot_path = os.path.join(beta_scan_dir, "beta_scan_analysis.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Beta scan plot saved: {plot_path}")
else:
    print("Need at least 2 beta runs to plot comparison")

## Mask Alignment Verification

In [ ]:
# Verify mask alignment with brain image
print("Verifying mask alignment...")

# Load mask
mask_nii = nib.load("data/masks/weightMatrix.nii")
mask_data = torch.from_numpy(mask_nii.get_fdata()).float()
mask_data = mask_data.permute(2, 1, 0)  # Apply same permutation as in loss

# Load a sample brain
sample_batch = next(iter(val_loader))
sample_brain = sample_batch['volume'][0, 0].cpu().numpy()  # [D, H, W]

# Create a temporary criterion to use smart_adjust_mask
temp_criterion = WeightedVAELoss("data/masks/weightMatrix.nii", beta=1e-5)
mask_adjusted = temp_criterion.smart_adjust_mask(
    mask_data.unsqueeze(0).unsqueeze(0), 
    sample_brain.shape
)[0, 0].cpu().numpy()

# Plot overlay on multiple slices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

slice_positions = [
    ('Axial', 0, sample_brain.shape[0]//2),
    ('Coronal', 1, sample_brain.shape[1]//2),
    ('Sagittal', 2, sample_brain.shape[2]//2)
]

for idx, (plane, axis, pos) in enumerate(slice_positions):
    # Get slices
    if axis == 0:
        brain_slice = sample_brain[pos, :, :]
        mask_slice = mask_adjusted[pos, :, :]
    elif axis == 1:
        brain_slice = sample_brain[:, pos, :]
        mask_slice = mask_adjusted[:, pos, :]
    else:
        brain_slice = sample_brain[:, :, pos]
        mask_slice = mask_adjusted[:, :, pos]
    
    # Brain only
    axes[0, idx].imshow(brain_slice, cmap='gray')
    axes[0, idx].set_title(f'{plane} - Brain', fontsize=12, fontweight='bold')
    axes[0, idx].axis('off')
    
    # Overlay
    axes[1, idx].imshow(brain_slice, cmap='gray')
    axes[1, idx].imshow(mask_slice, cmap='Reds', alpha=0.3)
    axes[1, idx].set_title(f'{plane} - Brain + Mask Overlay', fontsize=12, fontweight='bold')
    axes[1, idx].axis('off')

plt.tight_layout()
overlay_path = os.path.join(output_dir, "mask_alignment_verification.png")
plt.savefig(overlay_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Mask alignment verification saved: {overlay_path}")
print("\nCheck if mask aligns properly with brain structures (no flipping)")